In [ ]:
from imageloader import ImageLoader
import os
import cv2
import numpy as np
from numpy import load, zeros, ones, asarray
import torch, gc
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from matplotlib import pyplot
from tqdm import tqdm
import pandas as pd
import torch.nn.functional as F

In [ ]:
train  = "training_datasets"

In [ ]:
gc.collect()
torch.cuda.empty_cache()

loader = ImageLoader(train_dir=train, n=330)

# Load actual image data and input tensors
lr_images, hr_images, lr_shape, hr_shape = loader.load_image_as_tensors()
print("lr_images:", lr_images.shape)   
print("hr_images:", hr_images.shape)

# Ensure NumPy arrays are converted to PyTorch tensors
if not isinstance(lr_images, torch.Tensor):
    lr_images = torch.from_numpy(lr_images).float()
    hr_images = torch.from_numpy(hr_images).float()
    
# Move tensors to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from srgenerator256x3 import srgenerator as generator
from discriminator256x3 import D32x3 as discriminator

In [ ]:
# Initialize the generator
generator_model = generator(n_resblocks=16)
generator_model = generator_model.to(device)

# Function to build and initialize discriminators
def build_and_compile(D_model, input_shape):
    model = D_model()
    model = model.to(device)
    model.optimizer = optim.Adam(model.parameters(), lr=0.0002, betas=(0.5, 0.999))
    return model

# Build discriminator models
discriminator_model = build_and_compile(discriminator, hr_shape)

# Define loss functions
adversarial_loss = nn.BCEWithLogitsLoss()
pixelwise_loss = nn.L1Loss()

In [ ]:
 # Function to save loss history to Excel
def write_to_excel(file_path, generator_losses, discriminator_losses):
    df = pd.DataFrame({
        "Epoch": list(range(1, len(generator_losses) + 1)),
        "Generator Loss": generator_losses,
        "Discriminator Loss": discriminator_losses
    })
    df.to_excel(file_path, index=False)

In [ ]:
from torch.cuda.amp import autocast, GradScaler

def train_patchsrgan(discriminator, d_dims, lr_images, hr_images, generator_model):
    epochs = 100
    batch_size = 1
    save_interval = 2
    
    loss_file = f"patchsrgan{d_dims}.xlsx"
    
    train_dataset = TensorDataset(lr_images, hr_images)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    

    generator_optimizer = optim.Adam(generator_model.parameters(), lr=0.0002, betas=(0.5, 0.999))
    discriminator_optimizer = discriminator.optimizer

    # Mixed precision scaler
    scaler = GradScaler()

    g_losses_all, d_losses_all = [], []

    for e in range(epochs):
        g_losses, d_losses = [], []
        print(f"Starting epoch {e+1}/{epochs}")
        
        with tqdm(total=len(train_dataloader), desc=f"Epoch {e+1}/{epochs}", unit="batch") as pbar:
            for lr_imgs, hr_imgs in train_dataloader:
                lr_imgs = lr_imgs.to(device)
                hr_imgs = hr_imgs.to(device)

               #First forward pass
                with autocast(dtype=torch.float16):
                    fake_imgs = generator_model(lr_imgs)

                #Train discriminator_model

                discriminator.train()
                discriminator_optimizer.zero_grad()

                with autocast(dtype=torch.float16):
                    real_output = discriminator(hr_imgs)
                    fake_output = discriminator(fake_imgs.detach())

                    real_label = torch.ones_like(real_output)
                    fake_label = torch.zeros_like(fake_output)

                    d_loss_real = adversarial_loss(real_output, real_label)
                    d_loss_fake = adversarial_loss(fake_output, fake_label)
                    d_loss = 0.5 * (d_loss_real + d_loss_fake)

                # backward prop
                scaler.scale(d_loss).backward()
                scaler.step(discriminator_optimizer)
                scaler.update()

                # Train generator with discriminator_model frozen
                generator_optimizer.zero_grad(set_to_none=True)
                discriminator.eval()
                
                for p in discriminator.parameters():
                    p.requires_grad = False  # freeze discriminator_model weights

                with autocast(dtype=torch.float16):
                    fake_validity = discriminator(fake_imgs)
                    adv_loss = adversarial_loss(fake_validity, torch.ones_like(fake_validity))
                    content_loss = pixelwise_loss(fake_imgs, hr_imgs)
                    g_loss = content_loss + 0.02*adv_loss

                scaler.scale(g_loss).backward()
                scaler.step(generator_optimizer)
                scaler.update()

                # Unfreeze discriminator_mode for next iteration
                for p in discriminator.parameters():
                    p.requires_grad = True

                #  Logging/loss history
                g_losses.append(g_loss.item())
                d_losses.append(d_loss.item())
                pbar.update(1)

        g_loss_avg = np.mean(g_losses)
        d_loss_avg = np.mean(d_losses)
        g_losses_all.append(g_loss_avg)
        d_losses_all.append(d_loss_avg)

        print(f"Epoch {e+1}: G_loss={g_loss_avg:.5f}, D_loss={d_loss_avg:.5f}")
        write_to_excel(loss_file, g_losses_all, d_losses_all)

        if (e + 1) % save_interval == 0:
            torch.save({
                'generator_state_dict': generator_model.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'generator_optimizer': generator_optimizer.state_dict(),
                'discriminator_optimizer': discriminator_optimizer.state_dict(),
                'epoch': e,
            }, f"patchsrgan3D{d_dims}_{e+1}.pth")


In [ ]:
train_patchsrgan(discriminator_model, "", lr_images, hr_images, generator_model)